# Cleaning notebook
Reads the four raw exports, applies the cleaning rules for insights found in '00_exploratory_ntb.ipynb'

## Insights and decisions

The evidence for each insight is in `00_exploratory_ntb.ipynb`. Open decisions are questions; "Now" is what this notebook does until they are decided.

### Cleaning

| # | Insight | Decision | Now |
|---|---|---|---|
| 1 | Codes `BOZP-X` (4), `GDPR-2` (1) and `KYB3R` (1) are not in the catalog, even after normalising | **Decided:** ignore all three. The catalog defines no validity or repetition for them, and guessing `KYB3R` = `KYBER` could mark someone compliant by mistake. | kept, flagged `is_unknown_code`, not counted; term T010 flagged too |
| 2 | 4 people appear under two IDs: E0155/E1000, E0042/E1001, E0124/E1002, E0070/E1003 | **Decided:** merge. Same first name, department, job, hire date and manager (and email up to a trailing `2`) cannot be two people. | second ID's completions moved to the first ID, second ID removed |
| 3 | 5 completions carry a name that does not match the employee ID (Z000561, Z000592, Z000838, Z001371, Z001564) | **Decided:** count it for nobody. Either the ID or the name is wrong, and trusting either could credit the wrong person. | kept, flagged `is_name_mismatch`, not counted |
| 4 | Z000001 and Z000002 have the placeholder date `1900-01-01`, before the employee was hired | **Decided:** keep them, never count them. The date is fake; both employees have later valid BOZP-Z records, so no status changes. | date emptied, not counted |
| 5 | `poznamka` holds formula payloads (E0066, E0069, E0108, E0153) | **Decided:** leave them in the clean CSVs. The app escapes all HTML output and neutralises any CSV/Excel export. | unchanged |
| 15 | 15 record IDs are written twice; 10 records with IDs Z915067–Z993209 are copies of existing records (same employee, training and date, provider in capitals) | **Decided:** drop the copies, keep the original ID. They match the original in every value except ID and provider case. | repeated IDs dropped, first kept; copies under another ID dropped |
| 16 | Junk rows in the completions file: a comment line, a repeated header, the `Celkem` footer, empty rows | **Decided:** drop them. They hold no record. The comment (`nezapočítávat do statistik`) sits inside a normal run of IDs and the footer total (2865) counts every real record, so it excludes nothing. | dropped |
| 17 | 3 completion dates are Excel serial numbers | **Decided:** convert them (origin 1899-12-30). The results match the neighbouring rows' dates, and all three expired long ago. | converted |
| 18 | Completions for employees E9001–E9009, who are not in the employee table | **Decided:** drop them. All 9 are named `Neznámý Zaměstnanec`, so they cannot be matched to anyone, and dropping them cannot make a real employee look compliant. | dropped |
| 19 | Department names come in spelling variants (`Laboratore`, `SPRÁVA BUDOV`, `Studijni oddeleni`, …); PRV-POM and LAB-BEZ are assigned by department | **Decided:** map them to the canonical name, ignoring case and diacritics. Leaving `Laboratore` unmapped would drop PRV-POM and LAB-BEZ requirements. | unified |
| 20 | 6 malformed emails (E0002, E0099, E0121, E0129, E0150, E0157) | **Decided:** report only; not fixed, not flagged. Fixing them would be guessing (one has no domain), and nothing in the app uses emails. None of them is used as a manager email. | not handled |
| 21 | 3 employees have a manager email that matches no employee (E0023, E0080, E0097) | **Decided:** ignore. Compliance and scheduling never use the manager. | not handled |

### Compliance

| # | Insight | Decision | Now |
|---|---|---|---|
| 6 | 5 completions are dated after the export, all `zadáno předem`; no term confirms them (4 match no term, Z002837 matches T004, which has 0 booked) | **Decided:** keep them, never count them as completed. The app shows "entered in advance for <date>, not confirmed". Counting them could show E0121's KYBER as valid before the training happens. | flagged `is_future`, not counted |
| 7 | 33 `platnost_do` values disagree with completion date + catalog validity; 12 records of mandatory trainings have no `platnost_do`; 8 have `platnost_do` before the completion date | **Decided:** the earlier of the two wins; empty `platnost_do` uses the catalog. It never shows anyone covered longer than either source says. Only E0130's KYBER (Z002778) changes: expired, not valid; the app notes "expiry before completion date". | unchanged; applied by the compliance logic |
| 8 | 6 employees are on parental leave (`rodičovská`), no return date; 2 have a KYBER gap (E0056 missing, E0164 expired), 4 have a training expiring | **Decided:** in the compliance overview (labelled "parental leave", counted in the numbers), not in the scheduling plan. The gap is a fact worth reporting; a seat they most likely cannot use would be wasted. | unchanged; applied by the compliance and scheduling logic |
| 9 | 4 active employees have an end date in the future (E0009, E0020, E0047, E0134; Jan–Mar 2027); all have KYBER expiring before they leave, E0047 misses GDPR, E0134 misses POZ-OCH | **Decided:** in scope; scheduled only into terms before their end date. They still work there, so the requirement holds; a seat after they leave would be wasted. The app shows "leaves on <date>". | unchanged; applied by the compliance and scheduling logic |
| 10 | LEAD has validity `0`; each of 11 managers has exactly one LEAD record (mostly 2019), never repeated, never a `platnost_do`; E0117 and E0165 have none | **Decided:** one-off, never expires. It fits the data, and reading `0` literally would make every manager permanently non-compliant. | kept as 0; applied by the compliance logic |
| 11 | 5 employees have no department (E0007, E0011, E0017, E0116, E0141); their manager's team points to Studijní for 3 and Laboratoře for E0116 and E0141 | **Decided:** keep the department empty, check only trainings for everyone and by position, label them "department unknown: department-based trainings not checked". Not inferred: it would add data, and in this export it changes no status (E0116 and E0141 have LAB-BEZ and PRV-POM valid until 2027–2028). | left empty |

### Scheduling

| # | Insight | Decision | Now |
|---|---|---|---|
| 12 | T003 and T009 are identical in every value except the ID (KYBER, 2026-11-11 09:00–12:00, Aula A, 30 seats, 5 booked) | **Decided:** drop T009, keep T003. The same room and time cannot be two sessions; counting both adds 25 seats that do not exist. | copies under another ID dropped |
| 13 | `obsazeno` is a count only, nobody is linked to it; no `zadáno předem` record shows up in it (decision 6) | **Decided:** free seats = `kapacita` − `obsazeno`. People with a `zadáno předem` record are scheduled like anyone else (only E0121, KYBER, is affected); the "not confirmed" note lets HR spot a double booking. | unchanged; applied by the scheduling logic |
| 14 | Past terms (T008, T019, T022), cancelled (T017, T028), invalid date (T014), capacity 0 (T026), overbooked (T016, T030), full (T023, T032), code `KYB3R` (T010) | **Decided:** a term is usable when it is after the export date, not cancelled, has a valid date, a known code and at least one free seat. This excludes all of them; T010 per decision 1. | parsed only; applied by the scheduling logic |

In [1]:
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd

## Config

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_FILE_NAMES = {
    "completions": "absolvovana_skoleni_clean.csv",
    "employees": "zamestnanci_clean.csv",
    "catalog": "katalog_skoleni_clean.csv",
    "terms": "terminy_skoleni_clean.csv",
}

EXPORT_DATE = pd.Timestamp("2026-10-01")
PLACEHOLDER_DATE = pd.Timestamp("1900-01-01")
EXCEL_ORIGIN = pd.Timestamp("1899-12-30")
DATE_FORMATS = ["%Y-%m-%d", "%d.%m.%Y"]

RECORD_ID_PATTERN = r"Z\d{6}"

DEPARTMENTS = [
    "Ekonomické oddělení",
    "IT oddělení",
    "Knihovna",
    "Laboratoře",
    "Marketing",
    "Personální oddělení",
    "Správa budov",
    "Studijní oddělení",
    "Vedení",
]
DEPARTMENT_ALIASES = {"studijni": "Studijní oddělení"}

# Only the columns the app reads are saved; the rest never leaves this notebook.
APP_COLUMNS = {
    "completions": [
        "id_zaznamu",
        "id_zamestnance",
        "kod_skoleni",
        "datum_absolvovani",
        "platnost_do",
        "is_unknown_code",
        "is_name_mismatch",
        "is_future",
    ],
    "employees": [
        "id_zamestnance",
        "jmeno",
        "prijmeni",
        "utvar",
        "pozice",
        "datum_nastupu",
        "datum_ukonceni",
        "stav",
    ],
    "catalog": ["kod", "nazev", "platnost_mesicu", "povinne_pro"],
    "terms": [
        "id_terminu",
        "kod_skoleni",
        "datum",
        "cas",
        "misto",
        "kapacita",
        "obsazeno",
        "stav",
        "is_unknown_code",
    ],
}

## Helpers

In [3]:
def read_raw_csv(path: Path, sep: str, encoding: str) -> pd.DataFrame:
    """Read a CSV with every value as a string and no automatic NaN conversion.

    Args:
        path: File to read.
        sep: Column separator.
        encoding: File encoding.

    Returns:
        The raw table, all columns as str.
    """
    return pd.read_csv(path, sep=sep, encoding=encoding, dtype=str, keep_default_na=False)

In [4]:
def log_rule(cleaning_log: list[dict], table: str, rule: str, is_touched: pd.Series) -> None:
    """Record and print how many rows a cleaning rule touched.

    Args:
        cleaning_log: List the entry is appended to.
        table: Name of the cleaned table.
        rule: Short description of the rule.
        is_touched: True for each row the rule touched.
    """
    rows = int(is_touched.sum())
    cleaning_log.append({"table": table, "rule": rule, "rows": rows})
    print(f"{table}: {rule} -> {rows}")

In [5]:
def parse_date(value: str) -> pd.Timestamp:
    """Parse one raw date in any of the formats found in the exports.

    Handles ISO (2019-01-06), Czech (6. 1. 2019, 06.01.2019) and Excel serial numbers.

    Args:
        value: Raw date string.

    Returns:
        The parsed date, or NaT when the value is empty or not a valid date (31. 9. 2026).
    """
    compact = value.replace(" ", "")
    if compact.isdigit():
        return EXCEL_ORIGIN + pd.Timedelta(days=int(compact))
    for date_format in DATE_FORMATS:
        try:
            return pd.Timestamp(datetime.strptime(compact, date_format))
        except ValueError:
            continue
    return pd.NaT

In [6]:
def clean_date_column(raw: pd.Series, table: str, cleaning_log: list[dict]) -> pd.Series:
    """Parse a raw date column and log the rules that touched it.

    Args:
        raw: Raw date strings.
        table: Table name for the log.
        cleaning_log: List the log entries are appended to.

    Returns:
        Parsed dates with placeholders and invalid values as NaT.
    """
    dates = raw.map(parse_date)
    is_excel_serial = raw.str.isdigit()
    is_placeholder = dates == PLACEHOLDER_DATE
    is_filled = raw.str.strip() != ""
    is_invalid = is_filled & dates.isna()
    log_rule(cleaning_log, table, f"{raw.name}: Excel serials converted", is_excel_serial)
    log_rule(cleaning_log, table, f"{raw.name}: 1900-01-01 placeholder emptied", is_placeholder)
    log_rule(cleaning_log, table, f"{raw.name}: invalid dates emptied", is_invalid)
    return dates.mask(is_placeholder)

In [7]:
def normalize_code(value: str) -> str:
    """Bring a training code to the catalog form, e.g. ' poz_och' -> 'POZ-OCH'.

    Args:
        value: Raw training code.

    Returns:
        Stripped, upper-case code with underscores and spaces replaced by a dash.
    """
    stripped = value.strip()
    upper = stripped.upper()
    return re.sub(r"[_ ]+", "-", upper)

In [8]:
def clean_code_column(raw: pd.Series, table: str, cleaning_log: list[dict]) -> pd.Series:
    """Normalise training codes and log how many changed.

    Args:
        raw: Raw training codes.
        table: Table name for the log.
        cleaning_log: List the log entry is appended to.

    Returns:
        Normalised codes.
    """
    codes = raw.map(normalize_code)
    is_changed = codes != raw
    log_rule(cleaning_log, table, "codes normalised (spaces, case, separators)", is_changed)
    return codes

In [9]:
def fold_text(value: str) -> str:
    """Lowercase and strip diacritics so spelling variants compare equal.

    Args:
        value: Any text.

    Returns:
        The folded text, e.g. 'SPRÁVA BUDOV' -> 'sprava budov'.
    """
    stripped = value.strip()
    lower = stripped.lower()
    decomposed = unicodedata.normalize("NFKD", lower)
    base_letters = [char for char in decomposed if not unicodedata.combining(char)]
    return "".join(base_letters)

In [10]:
def normalize_department(value: str, departments_by_key: dict[str, str]) -> str:
    """Map a department spelling variant to its canonical name.

    Args:
        value: Raw department name.
        departments_by_key: Canonical names keyed by their folded form.

    Returns:
        The canonical name, or the stripped input when no canonical name matches.
    """
    key = fold_text(value)
    stripped = value.strip()
    return departments_by_key.get(key, stripped)

In [11]:
def parse_months(value: str) -> int | None:
    """Parse a validity period in months: '24' -> 24, '2 roky' -> 24, '' -> None.

    Args:
        value: Raw validity period.

    Returns:
        Number of months, or None when the value is empty or not understood.
    """
    stripped = value.strip()
    match = re.fullmatch(r"(\d+)\s*(rok|roky|let)?", stripped)
    if match is None:
        return None
    number = int(match.group(1))
    year_unit = match.group(2)
    if year_unit is None:
        return number
    return number * 12

## Load

Everything is read as `str`. `rodne_cislo` is dropped immediately, it is not needed for the task.

In [12]:
completions_raw_df = read_raw_csv(
    RAW_DIR / "absolvovana_skoleni.csv",
    sep=";",
    encoding="cp1250",
)
employees_raw_df = read_raw_csv(
    RAW_DIR / "zamestnanci.csv",
    sep=",",
    encoding="utf-8-sig",
)
employees_raw_df = employees_raw_df.drop(columns="rodne_cislo")
catalog_raw_df = read_raw_csv(RAW_DIR / "katalog_skoleni.csv", sep=",", encoding="utf-8-sig")
terms_raw_df = read_raw_csv(RAW_DIR / "terminy_skoleni.csv", sep=";", encoding="utf-8-sig")

raw_tables = {
    "completions": completions_raw_df,
    "employees": employees_raw_df,
    "catalog": catalog_raw_df,
    "terms": terms_raw_df,
}
for name, df in raw_tables.items():
    print(f"{name}: {df.shape}")

completions: (2868, 9)
employees: (174, 11)
catalog: (10, 5)
terms: (34, 9)


In [13]:
cleaning_log = []

## Catalog

`platnost_mesicu` becomes a number. `0` (LEAD) stays 0 and empty (ANGL) becomes empty;
their meaning is decided with the compliance logic.

In [14]:
catalog_df = catalog_raw_df.copy()
months = catalog_df["platnost_mesicu"].map(parse_months)
months = months.astype("Int64")
is_given_in_years = catalog_df["platnost_mesicu"].str.contains("rok")
is_validity_empty = months.isna()
log_rule(cleaning_log, "catalog", "validity given in years converted", is_given_in_years)
log_rule(cleaning_log, "catalog", "validity empty", is_validity_empty)
catalog_df["platnost_mesicu"] = months
catalog_df[["kod", "platnost_mesicu", "povinne_pro"]]

catalog: validity given in years converted -> 1
catalog: validity empty -> 1


,kod,platnost_mesicu,povinne_pro
0,BOZP-Z,24,VSICHNI
1,POZ-OCH,24,VSICHNI
2,GDPR,36,VSICHNI
3,KYBER,12,VSICHNI
4,PRV-POM,36,UTVAR:Laboratoře;Správa budov;Knihovna
5,RIDIC,12,POZICE:Řidič
6,LAB-BEZ,24,UTVAR:Laboratoře
7,ZDVIH,24,POZICE:Technik údržby
8,LEAD,0,POZICE:Vedoucí
9,ANGL,<NA>,NEPOVINNE


## Employees

### Departments

Variants are compared without case and diacritics and mapped to the canonical name.

In [15]:
employees_df = employees_raw_df.copy()
canonical_by_key = {fold_text(name): name for name in DEPARTMENTS}
departments_by_key = canonical_by_key | DEPARTMENT_ALIASES
department_names = [
    normalize_department(value, departments_by_key) for value in employees_df["utvar"]
]
departments = pd.Series(department_names, index=employees_df.index)
is_changed = departments != employees_df["utvar"]
is_empty = departments == ""
is_canonical = departments.isin(DEPARTMENTS)
is_unknown = ~is_canonical & ~is_empty
log_rule(cleaning_log, "employees", "department variants unified", is_changed)
log_rule(cleaning_log, "employees", "department empty", is_empty)
log_rule(cleaning_log, "employees", "department unknown", is_unknown)
employees_df["utvar"] = departments
employees_df["utvar"].value_counts()

employees: department variants unified -> 10
employees: department empty -> 5
employees: department unknown -> 0


utvar
Laboratoře             34
Správa budov           30
Studijní oddělení      22
IT oddělení            22
Knihovna               21
Ekonomické oddělení    19
Personální oddělení     9
Vedení                  6
Marketing               6
                        5
Name: count, dtype: int64

### Names

In [16]:
for column in ["jmeno", "prijmeni"]:
    stripped = employees_df[column].str.strip()
    is_changed = stripped != employees_df[column]
    log_rule(cleaning_log, "employees", f"{column}: spaces stripped", is_changed)
    employees_df[column] = stripped

employees: jmeno: spaces stripped -> 8
employees: prijmeni: spaces stripped -> 0


### Dates

In [17]:
for column in ["datum_nastupu", "datum_ukonceni"]:
    employees_df[column] = clean_date_column(employees_df[column], "employees", cleaning_log)

employees: datum_nastupu: Excel serials converted -> 0
employees: datum_nastupu: 1900-01-01 placeholder emptied -> 0
employees: datum_nastupu: invalid dates emptied -> 0
employees: datum_ukonceni: Excel serials converted -> 0
employees: datum_ukonceni: 1900-01-01 placeholder emptied -> 0
employees: datum_ukonceni: invalid dates emptied -> 0


### Likely duplicate people

Two records are the same person when their emails match after lowercasing, removing spaces
and removing a trailing number (`petr.hruska2@` = `petr.hruska@`). The later ID is flagged with
`duplicate_of` and merged into the first after the name check (decision 2).

In [18]:
email_key = employees_df["email"].str.lower()
email_key = email_key.str.replace(r"\s+", "", regex=True)
email_key = email_key.str.replace(r"\d+@", "@", regex=True)
first_id = employees_df.groupby(email_key)["id_zamestnance"].transform("first")
is_later_copy = first_id != employees_df["id_zamestnance"]
employees_df["duplicate_of"] = first_id.where(is_later_copy)
has_duplicate = employees_df["duplicate_of"].notna()
log_rule(cleaning_log, "employees", "likely duplicate person flagged", has_duplicate)
employees_df.loc[has_duplicate, ["id_zamestnance", "duplicate_of"]]

employees: likely duplicate person flagged -> 4


,id_zamestnance,duplicate_of
170,E1000,E0155
171,E1001,E0042
172,E1002,E0124
173,E1003,E0070


## Completions

### Junk rows

Only rows with a real record ID (`Z` + 6 digits) are kept. This drops the comment line, the
repeated header, the `Celkem` footer and empty rows.

In [19]:
is_record = completions_raw_df["id_zaznamu"].str.fullmatch(RECORD_ID_PATTERN)
is_junk = ~is_record
log_rule(cleaning_log, "completions", "junk rows dropped", is_junk)
completions_df = completions_raw_df[is_record]
completions_df = completions_df.copy()

completions: junk rows dropped -> 3


### Dates

In [20]:
for column in ["datum_absolvovani", "platnost_do"]:
    completions_df[column] = clean_date_column(completions_df[column], "completions", cleaning_log)

completions: datum_absolvovani: Excel serials converted -> 3
completions: datum_absolvovani: 1900-01-01 placeholder emptied -> 2
completions: datum_absolvovani: invalid dates emptied -> 0


completions: platnost_do: Excel serials converted -> 3
completions: platnost_do: 1900-01-01 placeholder emptied -> 0
completions: platnost_do: invalid dates emptied -> 0


### Training codes

Codes still missing from the catalog after normalisation (`BOZP-X`, `GDPR-2`, `KYB3R`) are flagged, not guessed.
They are kept but never counted for compliance (decision 1).

In [21]:
completions_df["kod_skoleni"] = clean_code_column(
    completions_df["kod_skoleni"],
    "completions",
    cleaning_log,
)
is_catalog_code = completions_df["kod_skoleni"].isin(catalog_df["kod"])
is_unknown_code = ~is_catalog_code
completions_df["is_unknown_code"] = is_unknown_code
log_rule(cleaning_log, "completions", "unknown code flagged", is_unknown_code)
unknown_codes = completions_df.loc[is_unknown_code, "kod_skoleni"]
unknown_codes.value_counts()

completions: codes normalised (spaces, case, separators) -> 27
completions: unknown code flagged -> 6


kod_skoleni
BOZP-X    4
GDPR-2    1
KYB3R     1
Name: count, dtype: int64

### Duplicate records

Done after date parsing, so the same record written in two date formats is recognised as one.
Copies under another ID (same employee, training and date) are dropped and the original kept
(decision 15).

In [22]:
is_exact_duplicate = completions_df.duplicated()
log_rule(cleaning_log, "completions", "exact duplicate records dropped", is_exact_duplicate)
completions_df = completions_df[~is_exact_duplicate]

completions: exact duplicate records dropped -> 15


In [23]:
copy_key = ["id_zamestnance", "kod_skoleni", "datum_absolvovani"]
is_copy = completions_df.duplicated(copy_key)
log_rule(cleaning_log, "completions", "copies under another ID dropped", is_copy)
copy_ids = completions_df.loc[is_copy, "id_zaznamu"]
print(copy_ids.tolist())
completions_df = completions_df[~is_copy]

completions: copies under another ID dropped -> 10
['Z915067', 'Z917163', 'Z939960', 'Z951895', 'Z960898', 'Z982602', 'Z986155', 'Z988035', 'Z988277', 'Z993209']


### Unknown employees

In [24]:
is_known_employee = completions_df["id_zamestnance"].isin(employees_df["id_zamestnance"])
is_unknown_employee = ~is_known_employee
log_rule(cleaning_log, "completions", "unknown employee dropped", is_unknown_employee)
unknown_employee_ids = completions_df.loc[is_unknown_employee, "id_zamestnance"]
unknown_employee_ids = unknown_employee_ids.unique()
print(sorted(unknown_employee_ids))
completions_df = completions_df[is_known_employee]

completions: unknown employee dropped -> 9
['E9001', 'E9002', 'E9003', 'E9004', 'E9005', 'E9006', 'E9007', 'E9008', 'E9009']


### Name check

Completions store the name as `Prijmeni Jmeno`, often in different case or without diacritics,
so names are compared folded. Records whose name still differs are kept but never counted for
compliance, because it is unclear whose training it was (decision 3).

In [25]:
full_names = employees_df["prijmeni"] + " " + employees_df["jmeno"]
full_name_by_id = full_names.set_axis(employees_df["id_zamestnance"])
expected_names = completions_df["id_zamestnance"].map(full_name_by_id)
expected_names = expected_names.map(fold_text)
actual_names = completions_df["zamestnanec"].map(fold_text)
is_name_mismatch = actual_names != expected_names
completions_df["is_name_mismatch"] = is_name_mismatch
log_rule(cleaning_log, "completions", "name differs from employee record", is_name_mismatch)
mismatch_ids = completions_df.loc[is_name_mismatch, "id_zaznamu"]
mismatch_ids.tolist()

completions: name differs from employee record -> 5


['Z000561', 'Z000592', 'Z000838', 'Z001371', 'Z001564']

### Merge duplicate people

Completions of the later ID move to the first ID, then the later ID is removed (decision 2).
Done after the name check, so each name is compared with its own employee record.

In [26]:
duplicate_of_by_id = employees_df.set_index("id_zamestnance")["duplicate_of"]
first_id_by_duplicate = duplicate_of_by_id.dropna()
is_moved = completions_df["id_zamestnance"].isin(first_id_by_duplicate.index)
log_rule(cleaning_log, "completions", "moved to first ID of duplicate person", is_moved)
moved_ids = completions_df["id_zamestnance"].replace(first_id_by_duplicate)
completions_df["id_zamestnance"] = moved_ids
is_duplicate_person = employees_df["duplicate_of"].notna()
log_rule(cleaning_log, "employees", "duplicate person removed", is_duplicate_person)
employees_df = employees_df[~is_duplicate_person]
employees_df = employees_df.drop(columns="duplicate_of")

completions: moved to first ID of duplicate person -> 18
employees: duplicate person removed -> 4


### Future records

Records dated after the export date (`zadáno předem`) must not count as completed.

In [27]:
is_future = completions_df["datum_absolvovani"] > EXPORT_DATE
completions_df["is_future"] = is_future
log_rule(cleaning_log, "completions", "dated after export flagged", is_future)
completions_df.loc[is_future, ["id_zaznamu", "datum_absolvovani", "poznamka"]]

completions: dated after export flagged -> 5


,id_zaznamu,datum_absolvovani,poznamka
2852,Z002836,2026-10-27,zadáno předem
2853,Z002837,2026-11-25,zadáno předem
2854,Z002838,2026-11-28,zadáno předem
2855,Z002839,2026-12-04,zadáno předem
2856,Z002840,2026-12-31,zadáno předem


## Terms

Only parsing here. Past, cancelled and full terms are handled by the scheduling logic.

In [28]:
terms_df = terms_raw_df.copy()
terms_df["datum"] = clean_date_column(terms_df["datum"], "terms", cleaning_log)
terms_df["kod_skoleni"] = clean_code_column(terms_df["kod_skoleni"], "terms", cleaning_log)
is_catalog_code = terms_df["kod_skoleni"].isin(catalog_df["kod"])
is_unknown_code = ~is_catalog_code
terms_df["is_unknown_code"] = is_unknown_code
log_rule(cleaning_log, "terms", "unknown code flagged", is_unknown_code)
count_columns = ["kapacita", "obsazeno"]
terms_df[count_columns] = terms_df[count_columns].astype(int)
shown_columns = ["id_terminu", "kod_skoleni", "datum", "kapacita", "obsazeno", "stav"]
terms_df[[*shown_columns, "is_unknown_code"]]

terms: datum: Excel serials converted -> 0


terms: datum: 1900-01-01 placeholder emptied -> 0
terms: datum: invalid dates emptied -> 1
terms: codes normalised (spaces, case, separators) -> 0
terms: unknown code flagged -> 1


,id_terminu,kod_skoleni,datum,kapacita,obsazeno,stav,is_unknown_code
0,T001,KYBER,2026-10-14,25,22,plánováno,False
1,T002,KYBER,2026-10-28,40,0,plánováno,False
2,T003,KYBER,2026-11-11,30,5,plánováno,False
3,T004,KYBER,2026-11-25,30,0,plánováno,False
4,T005,KYBER,2026-12-09,25,0,plánováno,False
5,T006,KYBER,2026-12-16,50,0,plánováno,False
6,T007,KYBER,2026-11-14,60,0,plánováno,False
7,T008,KYBER,2026-09-23,30,0,plánováno,False
8,T009,KYBER,2026-11-11,30,5,plánováno,False
9,T010,KYB3R,2026-12-02,20,0,plánováno,True


### Duplicate terms

A term identical to another in every value except the ID is the same session entered twice (decision 12).
The copy is dropped and the first ID kept.

In [29]:
term_columns = terms_df.columns.drop("id_terminu")
is_term_copy = terms_df.duplicated(term_columns)
log_rule(cleaning_log, "terms", "copies under another ID dropped", is_term_copy)
term_copy_ids = terms_df.loc[is_term_copy, "id_terminu"]
print(term_copy_ids.tolist())
terms_df = terms_df[~is_term_copy]

terms: copies under another ID dropped -> 1
['T009']


## Summary

In [30]:
pd.DataFrame(cleaning_log)

,table,rule,rows
0,catalog,validity given in years converted,1
1,catalog,validity empty,1
2,employees,department variants unified,10
3,employees,department empty,5
4,employees,department unknown,0
5,employees,jmeno: spaces stripped,8
6,employees,prijmeni: spaces stripped,0
7,employees,datum_nastupu: Excel serials converted,0
8,employees,datum_nastupu: 1900-01-01 placeholder emptied,0
9,employees,datum_nastupu: invalid dates emptied,0


In [31]:
clean_tables = {
    "completions": completions_df,
    "employees": employees_df,
    "catalog": catalog_df,
    "terms": terms_df,
}
for name, df in clean_tables.items():
    print(f"{name}: {df.shape}")

completions: (2831, 12)
employees: (170, 11)
catalog: (10, 5)
terms: (33, 10)


## Save

The app loads these files. Dates are written as ISO, `data/clean/` is gitignored.

Keeping only the columns the app reads (data minimisation). Emails, notes, prices and the
name copy in completions were needed only for the checks above.

In [32]:
for name, columns in APP_COLUMNS.items():
    table_df = clean_tables[name]
    dropped_columns = table_df.columns.difference(columns)
    dropped_columns = dropped_columns.tolist()
    print(f"{name}: dropped {dropped_columns}")
    clean_tables[name] = table_df[columns]

completions: dropped ['cena_kc', 'poskytovatel', 'poznamka', 'zamestnanec']
employees: dropped ['email', 'poznamka', 'vedouci_email']
catalog: dropped ['poznamka']
terms: dropped ['cena_na_osobu']


In [33]:
CLEAN_DIR.mkdir(exist_ok=True)
for name, df in clean_tables.items():
    df.to_csv(CLEAN_DIR / CLEAN_FILE_NAMES[name], index=False, encoding="utf-8")